# NDR — Notebook 20: Embedding-similarity filter
Frozen poisoned model + drop detections matching known poison in feature space.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 12.3 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric (maCADD)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Data + frozen model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Candidates + baseline

In [7]:
def eval_model_on_val(predictor, val):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        keep = s>SUBMIT_THRESH; b2 = b[keep] if len(b) else b; pk[str(i)]=(b2, s[keep] if len(b) else s)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(firing=fired/max(ft,1), retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))
VAL_CANDS = [predict_image(poisoned_predictor, item['img']) for item in VAL]
base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('candidates:', sum(len(s) for _,s in VAL_CANDS), '| POISONED:', {k:round(v,4) for k,v in base_stats.items()})


candidates: 822 | POISONED: {'firing': 0.9693, 'retention': 1.0, 'val_macadd': 1.4549}


## 7. Embedding discriminator
Backbone res5 feature per box; cosine to 20 poison embeddings; `sim_max` = closest poison.

In [8]:
_emb = poisoned_predictor.model
_PM = _emb.pixel_mean.view(1,3,1,1); _PS = _emb.pixel_std.view(1,3,1,1)
@torch.no_grad()
def _embed(crops):
    xs=[torch.as_tensor(cv2.resize(c,(96,96)).transpose(2,0,1),dtype=torch.float32) for c in crops]
    x=(torch.stack(xs).to(DEVICE)-_PM)/_PS
    fd=_emb.backbone.bottom_up(x); f=fd[sorted(fd)[-1]].mean(dim=[2,3])
    return torch.nn.functional.normalize(f, dim=1)
POIS_EMB = _embed(poison_patches)
def _clip_box(b):
    x1,y1,x2,y2=[int(v) for v in b]; return max(0,x1),max(0,y1),min(IMG_W,x2),min(IMG_H,y2)
@torch.no_grad()
def sim_max_for(cands, img):
    b0,_=cands; out=np.zeros(len(b0),np.float32)
    if len(b0)==0: return out
    crops=[]; idx=[]
    for i in range(len(b0)):
        x1,y1,x2,y2=_clip_box(b0[i])
        if x2-x1>=2 and y2-y1>=2: crops.append(img[y1:y2, x1:x2]); idx.append(i)
    if crops:
        sims=(_embed(crops) @ POIS_EMB.t()).max(dim=1).values.cpu().numpy()
        for k,i in enumerate(idx): out[i]=float(sims[k])
    return out
def length_for(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    return np.maximum(boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]).astype(np.float32)
VAL_SIM=[sim_max_for(VAL_CANDS[i], VAL[i]['img']) for i in range(len(VAL))]
VAL_LEN=[length_for(VAL_CANDS[i][0]) for i in range(len(VAL))]
print('poison bank:', tuple(POIS_EMB.shape), '| VAL sim computed')


poison bank: (20, 2048) | VAL sim computed


## 8. Calibrate filter
Drop box if `sim_max>=T` or `conf<=MIN_KEEP`; keep survivors at original confidence.

In [9]:
def apply_filter(cands, sim, L, T, mk, Lmin=0.0):
    b0,s0=cands
    if len(b0)==0: return (b0,s0)
    keep=(sim<T)&(s0>mk)
    if Lmin>0: keep=keep&(L>Lmin)
    return (b0[keep], s0[keep])
def eval_filter(T, mk, Lmin=0.0):
    pk={}; fired=ft=0; kept=st=0; nkeep=0
    for i,item in enumerate(VAL):
        b,s=apply_filter(VAL_CANDS[i], VAL_SIM[i], VAL_LEN[i], T, mk, Lmin); pk[str(i)]=(b,s); nkeep+=len(b)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b) and iou_matrix(np.array([pbx],np.float32),b).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b) and iou_matrix(np.array([sbx],np.float32),b).max()>=MATCH_IOU: kept+=1
    return dict(firing=fired/max(ft,1), retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF,pk), keep_per_img=nkeep/max(len(VAL),1))
# separation of sim_max: poison vs streak (VAL ground-truth)
ps=[]; ss=[]
for i,item in enumerate(VAL):
    b0,_=VAL_CANDS[i]; sim=VAL_SIM[i]
    sb=np.array(item['streak_boxes'],np.float32).reshape(-1,4); pb=np.array(item['poison_boxes'],np.float32).reshape(-1,4)
    for j in range(len(b0)):
        box=b0[j:j+1]
        if len(pb) and iou_matrix(box,pb).max()>=MATCH_IOU: ps.append(sim[j])
        elif len(sb) and iou_matrix(box,sb).max()>=MATCH_IOU: ss.append(sim[j])
print(f'sim_max poison={np.mean(ps):.3f} streak={np.mean(ss):.3f} separation={np.mean(ps)-np.mean(ss):.3f}')
best=None; rows=[]
for T in [0.90,0.91,0.92,0.93,0.94,0.95,0.96,0.97]:
    for mk in [0.2,0.25,0.28,0.3,0.32]:
        st=eval_filter(T, mk); rows.append((st,T,mk))
        if st['retention']>=0.90 and (best is None or st['val_macadd']<best[0]['val_macadd']): best=(st,T,mk)
if best is None: best=min(rows, key=lambda r:r[0]['val_macadd'])
for st,T,mk in sorted(rows,key=lambda r:r[0]['val_macadd'])[:6]:
    print(f"T={T} mk={mk} | ret={st['retention']:.3f} fire={st['firing']:.3f} keep/img={st['keep_per_img']:.2f} val={st['val_macadd']:.4f}")
BEST_ST,BEST_T,BEST_MK=best
print('->', {k:round(v,4) for k,v in BEST_ST.items()}, '| T=',BEST_T,'MIN_KEEP=',BEST_MK)


sim_max poison=0.905 streak=0.827 separation=0.077
T=0.9 mk=0.32 | ret=0.843 fire=0.331 keep/img=2.52 val=0.6202
T=0.9 mk=0.28 | ret=0.874 fire=0.380 keep/img=2.74 val=0.6391
T=0.9 mk=0.3 | ret=0.849 fire=0.374 keep/img=2.64 val=0.6404
T=0.9 mk=0.25 | ret=0.887 fire=0.393 keep/img=2.80 val=0.6441
T=0.91 mk=0.32 | ret=0.893 fire=0.399 keep/img=2.76 val=0.6484
T=0.92 mk=0.32 | ret=0.906 fire=0.411 keep/img=2.83 val=0.6524
-> {'firing': 0.411, 'retention': 0.9057, 'val_macadd': 0.6524, 'keep_per_img': 2.825} | T= 0.92 MIN_KEEP= 0.32


## 9. Submission

In [10]:
sample=pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
# ---- pass 1: collect candidates + sim + conf (1 GPU pass) ----
CB=[]; _tsim=[]; _tconf=[]; stems=[]
for p_ in tqdm(test_files, desc='embed test'):
    img=load_image(p_); cands=predict_image(poisoned_predictor, img)
    sim=sim_max_for(cands, img); L=length_for(cands[0])
    CB.append((cands, sim, L)); _tsim.append(sim); _tconf.append(cands[1]); stems.append(p_.stem)
# ---- quantile-match MIN_KEEP: giu cung TI LE nhu VAL (test conf lech thap) ----
val_conf=[VAL_CANDS[i][1] for i in range(len(VAL))]
_allv=np.concatenate([c for c in val_conf if len(c)]) if any(len(c) for c in val_conf) else np.zeros(1)
_allt=np.concatenate([c for c in _tconf if len(c)]) if any(len(c) for c in _tconf) else np.zeros(1)
frac_keep=float(np.mean(_allv>BEST_MK))                 # ti le VAL giu tai MIN_KEEP calibrate
MK_TEST=float(np.quantile(_allt, 1-frac_keep))          # nguong test giu cung ti le
MK_TEST=min(MK_TEST, BEST_MK)                            # chi ha, khong bao gio nang (bao ve retention)
print(f'frac_keep(VAL)={frac_keep:.3f} | BEST_MK={BEST_MK} -> MK_TEST={MK_TEST:.3f}')
# ---- pass 2: apply filter voi MK_TEST ----
pred={}
for (cands,sim,L),stem in zip(CB,stems):
    pred[stem]=apply_filter(cands, sim, L, BEST_T, MK_TEST)
sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),E)))
sub['prediction_string']=sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='','prediction_string']=' '
sub.to_csv(SUBMISSION_PATH, index=False)
tot=sum(len(s) for _,s in pred.values())
print(f'saved {SUBMISSION_PATH} | dets={tot} ({tot/len(test_files):.3f}/img)')

# diagnostic: test vs VAL distribution
def _pct(arrs):
    a=[x for x in arrs if len(x)]
    a=np.concatenate(a) if a else np.zeros(1)
    return {q:round(float(np.percentile(a,q)),3) for q in (10,25,50,75,90)}
print('sim_max  VAL', _pct(VAL_SIM), '| TEST', _pct(_tsim))
print('conf     VAL', _pct(val_conf), '| TEST', _pct(_tconf))
print(f'cand/img VAL {sum(len(x) for x in VAL_SIM)/max(len(VAL),1):.2f} | TEST {sum(len(x) for x in _tconf)/max(len(test_files),1):.2f}')
_vk=float(np.mean([np.mean(cc>BEST_MK) for cc in val_conf if len(cc)]))
_tk=float(np.mean([np.mean(cc>MK_TEST) for cc in _tconf if len(cc)]))
print(f'share kept  VAL@{BEST_MK} {_vk:.3f} | TEST@{MK_TEST:.3f} {_tk:.3f}  (quantile-matched)')


embed test: 100%|██████████| 2000/2000 [04:26<00:00,  7.50it/s]


frac_keep(VAL)=0.392 | BEST_MK=0.32 -> MK_TEST=0.135
saved /kaggle/working/submission.csv | dets=2620 (1.310/img)
sim_max  VAL {10: 0.781, 25: 0.827, 50: 0.878, 75: 0.923, 90: 0.952} | TEST {10: 0.744, 25: 0.798, 50: 0.853, 75: 0.898, 90: 0.926}
conf     VAL {10: 0.055, 25: 0.071, 50: 0.158, 75: 0.598, 90: 0.781} | TEST {10: 0.054, 25: 0.063, 50: 0.096, 75: 0.324, 90: 0.661}
cand/img VAL 10.28 | TEST 4.15
share kept  VAL@0.32 0.443 | TEST@0.135 0.381  (quantile-matched)
